In [3]:
import sys
import json
import pandas as pd

sys.path.append("..")

from etl_pipeline_local import get_win_1x_table

In [ ]:
# Data upload - do not implement
# Bet setup
target_col = "dnb_1"

# Load data
df_loaded = get_win_1x_table()
df_loaded = df_loaded.sort_values('time').reset_index(drop=True)
df = df_loaded.copy()

In [5]:
def get_mask(df, params_dict):
    mask = pd.Series(True, index=df.index)
    features = set()

    for key in params_dict:
        for suffix in ["_cat", "_use_min", "_use_max", "_min", "_max", "_include_missing"]: # ,   "_min_idx", "_max_idx", 
             if key.endswith(suffix):
                features.add(key[:-len(suffix)])
                break

    for feat in features:
        s = df[feat]

        include_missing = params_dict.get(f"{feat}_include_missing", False)
        use_min = params_dict.get(f"{feat}_use_min", True)
        use_max = params_dict.get(f"{feat}_use_max", True)

        feat_mask = pd.Series(True, index=df.index)

        # Categorical filtering
        if f"{feat}_cat" in params_dict:
            feat_mask &= s.isin(params_dict[f"{feat}_cat"])

        # Numerical filtering
        if f"{feat}_min" in params_dict and use_min:
            feat_mask &= s >= params_dict[f"{feat}_min"]

        if f"{feat}_max" in params_dict and use_max:
            feat_mask &= s <= params_dict[f"{feat}_max"]

        if include_missing:
            feat_mask = feat_mask | s.isna()
        else:
            feat_mask = feat_mask & s.notna()

        if params_dict[f"use_{feat}"]:
            mask &= feat_mask

    return mask


def round_to_step(x, step):
    """
    Arrotonda x a un multiplo di 'step'.
    """
    try:
        if not step:
            return str(x)
        elif step <= 0:
            raise ValueError("step deve essere > 0")
        else:
            ratio = x / step
            return round(ratio) * step
    
    except Exception as e: 
        return None

In [6]:
with open("../../strategies/strategies.json", "r", encoding="utf-8") as f:
    data = json.load(f)

params_dict = data[target_col]['params_dict']
feature_bins_map = data[target_col]['feature_bins_map']

# Feature Engineering
df = df[(df["chance1x2_quote_current1"] >= 1.9) & (df["chance1x2_quote_current1"] <= 2.6)]

df_binned = df.copy()

for feat, step in feature_bins_map.items():
    df[feat] = [round_to_step(x, step) for x in df[feat]]

    if isinstance(step, int):
        df[feat] = df[feat].astype("Int64")


mask = get_mask(df, params_dict)

df_filtered = df[mask]
df_filtered.head()


,time,chance1x2_chance_p1,chance1x2_chance_px,chance1x2_chance_p2,chance1x2_chance_p1x,chance1x2_chance_p2x,chance1x2_chance_p12,chance1x2_chance_pHt1,chance1x2_chance_pHtx,chance1x2_chance_pHt2,...,underOver_flashback_under35,underOver_flashback_over35,team_goal,team_goalHt,team_corner,chance1x2_xg_home,chance1x2_xg_away,chance1x2_xg_total,chance1x2_xg,win_1x
42,2025-05-16 20:45:00,40.8,29.8,29.4,70.6,59.2,70.2,31.9,37.9,30.2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
106,2025-05-17 17:30:00,39.9,32.8,27.3,72.7,60.1,67.2,30.4,40.7,28.9,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
132,2025-05-17 21:00:00,52.0,31.1,16.9,83.1,48.0,68.9,36.1,43.4,20.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
250,2025-05-18 21:00:00,49.1,25.2,25.7,74.3,50.9,74.8,28.5,41.4,30.1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
436,2025-05-25 09:30:00,41.2,26.7,32.1,67.9,58.8,73.3,26.2,48.2,25.6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
